# [JAX·TPU 선택 심화] 00 · 라이브러리로 직접 준비하는 물체 이미지 데이터
**목표:** 어떤 라이브러리가 무엇을 하는지 확인하고 이미지 분류에 쓸 데이터를 직접 준비합니다.

Codespaces는 편집·데이터 준비·CPU 추론을 맡고 Colab은 GPU·TPU 학습을 맡습니다. 이 노트북은 원격 연산을 시작하지 않습니다. CIFAR-100의 32×32 공개 이미지로 흐름을 익힙니다. 실제 작업대 사진의 성능은 별도로 확인해야 합니다.

커널은 **Vision AI (Codespaces CPU)**를 선택하세요. 처음에는 `scripts/setup.sh`로 개별 라이브러리를 설치합니다. 설치 과정에서는 데이터를 준비하지 않으므로 아래 셀에서 다운로드·선별·저장 과정을 직접 실행합니다.

기본 HF·PyTorch 과정은 `hf_colab_gpu/notebooks`에 있습니다. 이 심화 과정은 프로젝트 최상위에서 `bash scripts/setup.sh --with-jax`로 준비합니다.

## 라이브러리를 직접 불러오기
가상환경은 라이브러리 버전을 구분하는 공간입니다. 아래 `import`가 이 노트북에서 실제로 사용하는 라이브러리입니다.

| 가져오는 이름 | 설치할 패키지 | 하는 일 |
|---|---|---|
| `numpy` | `numpy` | 이미지 배열, 라벨, `.npz` 파일 |
| `PIL.Image` | `Pillow` | 이미지 읽기와 크기 조절 |
| `matplotlib.pyplot` | `matplotlib` | 이미지와 그래프 표시 |
| `IPython` | `ipykernel`과 함께 설치 | 노트북 안에 그래프 표시 |

`pathlib`, `os`, `sys`, `json`, `hashlib`, `importlib.metadata`는 Python 표준 라이브러리이므로 따로 설치하지 않습니다. 필요한 추가 라이브러리는 사용하는 셀에서 직접 불러옵니다.

In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
from importlib.metadata import version

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/vision-ai")]
ROOT = next((path for path in candidates if (path / ".vision-lab-root").is_file()), None)
if ROOT is None:
    raise RuntimeError(".vision-lab-root가 있는 수업 폴더에서 열거나 Colab에 실습 파일을 먼저 업로드하세요.")
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
print("Project:", ROOT)
print("Python:", sys.executable)

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
for package in ("numpy", "Pillow", "matplotlib", "ipykernel"):
    print(f"{package}: {version(package)}")

## 1. Parquet를 읽는 pyarrow 불러오기
`pyarrow.parquet`는 표 형태의 Parquet 파일을 읽습니다. `urllib.request`는 파일을 내려받고 `io.BytesIO`는 압축된 이미지 바이트를 Pillow에 전달합니다. `io`, `shutil`, `urllib.request`는 표준 라이브러리입니다.

In [ ]:
import io
import shutil
import urllib.request
import pyarrow.parquet as pq

print("pyarrow:", version("pyarrow"))
CLASSES = ["bottle", "bowl", "can", "cup", "plate"]
FINE_LABEL_IDS = [9, 10, 16, 28, 61]
DATASET_ID = "uoft-cs/cifar100"
DATASET_REVISION = "aadb3af77e9048adbea6b47c21a81e47dd092ae5"
SOURCE_FILES = {
    "train": "694865d6b990e234804f01268586c41e88bcbbb75e20858432c05ad4081aca23",
    "test": "98776c529bb146a9c791229df74a5cf076be9b43d82dbbd334b6a7788d73dc68",
}
SEED = 42
TRAIN_PER_CLASS, VALIDATION_PER_CLASS, TEST_PER_CLASS = 100, 20, 40
CACHE = ROOT / ".cache/data"
DATA_PATH = ROOT / "data/prepared"
print("Classes:", dict(zip(FINE_LABEL_IDS, CLASSES)))

### 저장된 데이터 읽기와 무결성 확인
`np.load(..., allow_pickle=False)`로 이미지·라벨·ID를 읽습니다. SHA256과 분할별 ID를 검사해 다른 데이터가 섞이거나 손상된 경우 중단합니다. 이 함수의 본문도 아래에 모두 표시합니다.

In [ ]:
def digest(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

In [ ]:
def load_prepared(path):
    path = Path(path)
    manifest = json.loads((path / "manifest.json").read_text(encoding="utf-8"))
    classes = manifest["classes"]
    if len(classes) < 2 or len(set(classes)) != len(classes):
        raise ValueError("클래스 목록이 잘못됐습니다.")
    splits, all_ids = {}, set()
    for name in ("train", "validation", "test"):
        record = manifest["splits"][name]
        if record["file"] != f"{name}.npz":
            raise ValueError("데이터 파일 경로가 예상 형식과 다릅니다.")
        file = path / record["file"]
        if digest(file) != record["sha256"]:
            raise ValueError(f"{name} 데이터 체크섬 불일치")
        with np.load(file, allow_pickle=False) as a:
            images, labels, ids = a["images"], a["labels"], a["ids"].tolist()
        if images.dtype != np.uint8 or images.ndim != 4 or images.shape[-1] != 3:
            raise ValueError("images는 uint8 NHWC RGB여야 합니다.")
        if len(images) != len(labels) or len(ids) != len(labels) or len(labels) != record["count"]:
            raise ValueError("이미지·라벨·식별자 개수가 다릅니다.")
        if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or len(labels) == 0 or labels.min() < 0 or labels.max() >= len(classes):
            raise ValueError("라벨 범위가 잘못됐습니다.")
        if len(set(ids)) != len(ids) or all_ids.intersection(ids):
            raise ValueError("분할 간 이미지 ID 중복")
        all_ids.update(ids)
        splits[name] = {"images": images, "labels": labels, "ids": ids}
    identity = {k: manifest[k] for k in ("classes", "splits", "seed", "preprocess")}
    if hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest() != manifest["dataset_sha256"]:
        raise ValueError("데이터 manifest 지문 불일치")
    return splits, manifest

## 2. 기존 데이터가 있으면 먼저 검증하기
같은 조건으로 만든 파일은 재사용합니다. 다른 조건의 실험 폴더는 덮어쓰지 않습니다. 데이터 수나 seed를 바꾸려면 `DATA_PATH`를 새로운 폴더로 지정하세요.

In [ ]:
selection = {"train_per_class": TRAIN_PER_CLASS, "val_per_class": VALIDATION_PER_CLASS,
             "test_per_class": TEST_PER_CLASS}
REUSE_PREPARED = (DATA_PATH / "manifest.json").is_file()
if REUSE_PREPARED:
    splits, manifest = load_prepared(DATA_PATH)
    source = manifest.get("source", {})
    if (manifest["dataset_name"] != "cifar100_food_containers" or manifest["seed"] != SEED
            or manifest["classes"] != CLASSES or source.get("selection") != selection
            or source.get("revision") != DATASET_REVISION or source.get("source_sha256") != SOURCE_FILES):
        raise ValueError("다른 조건의 데이터가 있습니다. 새 DATA_PATH를 지정하세요.")
    print("기존 데이터의 SHA256과 실험 조건을 확인했습니다.")
elif DATA_PATH.exists() and any(DATA_PATH.iterdir()):
    raise FileExistsError("완성되지 않은 데이터 폴더가 있습니다. 내용을 확인하거나 새 DATA_PATH를 지정하세요.")

## 3. 원본 다운로드와 SHA256 확인
처음 실행할 때 원본 Parquet 두 개를 약 142 MB 내려받습니다. 이미 있는 파일도 SHA256을 확인합니다. 주소에 고정 revision을 넣어 다른 시점의 데이터를 받지 않도록 합니다.

In [ ]:
CACHE.mkdir(parents=True, exist_ok=True)
source_paths = {}
if not REUSE_PREPARED:
    for split_name in ("train", "test"):
        filename = f"{split_name}-00000-of-00001.parquet"
        target = CACHE / filename
        if not target.is_file():
            partial = target.with_suffix(".part")
            url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/cifar100/{filename}"
            print("Download:", filename)
            try:
                with urllib.request.urlopen(url, timeout=120) as response, partial.open("wb") as stream:
                    shutil.copyfileobj(response, stream)
                if digest(partial) != SOURCE_FILES[split_name]:
                    raise ValueError("다운로드한 파일의 SHA256이 다릅니다.")
                partial.replace(target)
            finally:
                partial.unlink(missing_ok=True)
        if digest(target) != SOURCE_FILES[split_name]:
            raise ValueError(f"캐시 SHA256 불일치: {target}")
        source_paths[split_name] = target
        print("Verified:", filename)
else:
    print("검증한 준비 데이터를 재사용하므로 원본 다운로드를 생략합니다.")

## 4. pyarrow로 표 읽기
필요한 `img`, `fine_label` 두 열만 읽습니다. 원본 학습 분할에는 50,000장, 테스트 분할에는 10,000장이 있어야 합니다.

In [ ]:
if not REUSE_PREPARED:
    train_table = pq.read_table(source_paths["train"], columns=["img", "fine_label"])
    test_table = pq.read_table(source_paths["test"], columns=["img", "fine_label"])
    train_labels = train_table["fine_label"].to_numpy()
    test_labels = test_table["fine_label"].to_numpy()
    assert len(train_labels) == 50000 and len(test_labels) == 10000
    print(train_table.schema)
    print("Original split sizes:", len(train_labels), len(test_labels))

## 5. NumPy로 클래스별 이미지를 고르기
원본 학습 분할에서 클래스마다 학습 100장·검증 20장을 서로 겹치지 않게 선택합니다. 테스트는 원본 테스트 분할에서 클래스마다 40장을 고릅니다. 검증 이미지는 epoch 선택에, 테스트 이미지는 최종 평가에 사용합니다.

In [ ]:
if not REUSE_PREPARED:
    if min(TRAIN_PER_CLASS, VALIDATION_PER_CLASS, TEST_PER_CLASS) < 1:
        raise ValueError("클래스별 선택 수는 양수여야 합니다.")
    rng = np.random.default_rng(SEED)
    train_indices, validation_indices = [], []
    for label in FINE_LABEL_IDS:
        candidates = np.flatnonzero(train_labels == label)
        if TRAIN_PER_CLASS + VALIDATION_PER_CLASS > len(candidates):
            raise ValueError("학습·검증 요청 수가 원본 클래스 수를 넘습니다.")
        shuffled = rng.permutation(candidates)
        train_indices.extend(shuffled[:TRAIN_PER_CLASS])
        validation_indices.extend(shuffled[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VALIDATION_PER_CLASS])
    rng = np.random.default_rng(SEED + 1)
    if TEST_PER_CLASS > 100:
        raise ValueError("원본 테스트에는 클래스마다 100장이 있습니다.")
    test_indices = np.concatenate([rng.permutation(np.flatnonzero(test_labels == label))[:TEST_PER_CLASS]
                                   for label in FINE_LABEL_IDS])
    assert not set(train_indices).intersection(validation_indices)
    print("Selected:", len(train_indices), len(validation_indices), len(test_indices))

## 6. Pillow로 이미지 바이트를 RGB 배열로 바꾸기
이 단계에서는 원본 32×32 픽셀을 보존합니다. 모델에 넣을 224×224 전처리는 다음 노트북에서 직접 수행합니다.

In [ ]:
if not REUSE_PREPARED:
    label_map = {old: new for new, old in enumerate(FINE_LABEL_IDS)}
    splits = {}
    for name, table, labels, indices, origin in [
        ("train", train_table, train_labels, train_indices, "train"),
        ("validation", train_table, train_labels, validation_indices, "train"),
        ("test", test_table, test_labels, test_indices, "test"),
    ]:
        pixels = []
        for row in table.take([int(index) for index in indices])["img"].to_pylist():
            with Image.open(io.BytesIO(row["bytes"])) as image:
                pixels.append(np.asarray(image.convert("RGB")))
        splits[name] = {"images": np.stack(pixels),
                        "labels": np.asarray([label_map[int(labels[index])] for index in indices], dtype=np.int64),
                        "ids": [f"cifar100/{origin}/{index:05d}" for index in indices]}
    del train_table, test_table
    print("Train pixels:", splits["train"]["images"].shape, splits["train"]["images"].dtype)

### 분할을 NPZ와 manifest로 저장하기
이미지 배열은 압축한 `.npz`, 출처·클래스·파일 지문은 `manifest.json`으로 저장합니다. 같은 픽셀이 두 분할에 있으면 데이터 누출을 막기 위해 중단합니다.

In [ ]:
PREPROCESS = {"size": 224, "mean": [0.5] * 3, "std": [0.5] * 3,
              "resize": "bilinear", "layout": "NHWC"}

In [ ]:
def save_prepared(output, splits, classes, source, seed):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    # Exact duplicate pixels across splits are excluded before this point.
    seen = {}
    for name, split in splits.items():
        for image in split["images"]:
            key = hashlib.sha256(image.tobytes()).hexdigest()
            if key in seen and seen[key] != name:
                raise ValueError(f"분할 간 중복 이미지: {seen[key]} / {name}")
            seen[key] = name
    records = {}
    for name, split in splits.items():
        path = output / f"{name}.npz"
        with path.with_suffix(".tmp").open("wb") as stream:
            np.savez_compressed(stream, images=split["images"], labels=split["labels"], ids=np.asarray(split["ids"], dtype=str))
        path.with_suffix(".tmp").replace(path)
        records[name] = {"file": path.name, "sha256": digest(path), "count": len(split["labels"]),
                         "per_class": {c: int(np.sum(split["labels"] == i)) for i, c in enumerate(classes)}}
    identity = {"classes": classes, "splits": records, "seed": seed, "preprocess": PREPROCESS}
    manifest = {"schema_version": 1, "dataset_name": source["name"], **identity, "source": source,
                "dataset_sha256": hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()}
    (output / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return manifest

In [ ]:
if not REUSE_PREPARED:
    source = {"name": "cifar100_food_containers", "dataset_id": DATASET_ID,
              "revision": DATASET_REVISION, "source_sha256": SOURCE_FILES,
              "original_url": "https://www.cs.toronto.edu/~kriz/cifar.html",
              "original_resolution": [32, 32], "selection": selection}
    manifest = save_prepared(DATA_PATH, splits, CLASSES, source, SEED)
splits, manifest = load_prepared(DATA_PATH)
classes = manifest["classes"]
for name, split in splits.items():
    print(name, dict(zip(classes, np.bincount(split["labels"], minlength=len(classes)).tolist())))
print("Dataset fingerprint:", manifest["dataset_sha256"])

## 7. Matplotlib으로 학습 이미지 확인
각 클래스에서 3장을 표시합니다. 작은 이미지를 확대해도 새로운 세부 정보가 생기지는 않습니다. 물체와 배경을 함께 살펴보세요.

In [ ]:
fig, axes = plt.subplots(len(classes), 3, figsize=(8, 2 * len(classes)), squeeze=False)
for label, name in enumerate(classes):
    indices = np.flatnonzero(splits["train"]["labels"] == label)[:3]
    for column, index in enumerate(indices):
        axes[label, column].imshow(splits["train"]["images"][index], interpolation="nearest")
        axes[label, column].set_title(f"{name} · sample {column + 1}")
        axes[label, column].axis("off")
fig.suptitle("Training images · CIFAR-100 food containers · native 32 × 32", y=1.01)
fig.tight_layout()
plt.show()

## 확인 질문과 다음 단계
- 컵과 그릇을 구분하기 어려운 사진에는 어떤 특징이 있나요?
- 같은 촬영 장면을 학습과 테스트에 복사하면 평가가 어떻게 달라질까요?
- 어떤 라이브러리가 파일 다운로드·이미지 변환·배열 저장을 각각 맡았나요?

다음은 `01_pretrained_inference.ipynb`입니다. 데이터 출처는 [CIFAR](https://www.cs.toronto.edu/~kriz/cifar.html)와 [고정 데이터 저장소](https://huggingface.co/datasets/uoft-cs/cifar100/tree/aadb3af77e9048adbea6b47c21a81e47dd092ae5)입니다.